## 1. Limpieza de datos

Importamos pandas y obtenemos el dataset mediante kagglehub.

Cargamos los archivos de personajes, pociones y hechizos en tres
DataFrames. Utilizamos los mismos nombres de variables que en la fase 1
para mantener la coherencia del proyecto.

In [98]:
import pandas as pd
import kagglehub
from pathlib import Path

# Obtenemos la carpeta del dataset.
path = kagglehub.dataset_download(
    "gulsahdemiryurek/harry-potter-dataset"
)

# Leemos los CSV, cuyos campos están separados por punto y coma.
df_personajes = pd.read_csv(Path(path) / "Characters.csv", sep=";")
df_pociones = pd.read_csv(Path(path) / "Potions.csv", sep=";")
df_hechizos = pd.read_csv(Path(path) / "Spells.csv", sep=";")

Using Colab cache for faster access to the 'harry-potter-dataset' dataset.



## Pregunta 1 ¿Qué casa de Hogwarts tiene más personajes registrados?

Primero consultamos los valores distintos de House para conocer
las categorías existentes y comprobar cómo están escritas.

Seleccionamos la columna con ["House"] y utilizamos unique()
para obtener sus valores sin repetirlos.

In [99]:
# Mostramos los valores distintos de la columna House.
df_personajes["House"].unique()

array(['Gryffindor', 'Ravenclaw', 'Slytherin', nan, 'Hufflepuff',
       'Beauxbatons Academy of Magic', 'Durmstrang Institute'],
      dtype=object)

La columna House contiene las cuatro casas de Hogwarts, pero también
dos escuelas de magia y valores ausentes.

Para comparar únicamente las casas de Hogwarts, filtramos los personajes
que pertenecen a Gryffindor, Ravenclaw, Slytherin o Hufflepuff.

Guardamos la selección en un nuevo DataFrame llamado df_hogwarts.
`.isin(casas_hogwarts)` Devuelve True cuando el valor pertenece a esa lista y False cuando no.

La tabla original se conserva.

In [100]:
# Creamos una lista con las cuatro casas que queremos comparar.
casas_hogwarts = [
    "Gryffindor",
    "Ravenclaw",
    "Slytherin",
    "Hufflepuff"
]

# Seleccionamos los personajes cuya casa está en la lista.
df_hogwarts = df_personajes[
    
    df_personajes["House"].isin(casas_hogwarts)
]

Utilizamos value_counts() para contar cuántos personajes registrados
pertenecen a cada casa de Hogwarts.

Por defecto, este método ordena los resultados de mayor a menor
frecuencia. La primera casa será la que tenga más personajes
en los datos seleccionados.

In [101]:
# Contamos los personajes de cada casa y guardamos el resultado.
personajes_por_casa = df_hogwarts["House"].value_counts()

# Mostramos el conteo.
personajes_por_casa

,count
House,
Gryffindor,38
Slytherin,28
Ravenclaw,18
Hufflepuff,13


## Pregunta 2: ¿Cómo se distribuyen los personajes por género dentro de cada casa?

Queremos comprobar cuántos personajes de cada género hay registrados
en las cuatro casas de Hogwarts.

Antes de agruparlos, revisamos los valores distintos de Gender
en el DataFrame df_hogwarts.

In [102]:
# Consultamos los géneros registrados entre los personajes seleccionados.
df_hogwarts["Gender"].unique()

array(['Male', 'Female', nan], dtype=object)

Agrupamos los personajes por House y Gender, y contamos cuántos
registros hay en cada combinación.

Utilizamos `dropna=False` para incluir los personajes cuyo género
no está registrado.

`.size()`	Cuenta cuántas filas hay en cada grupo.

In [103]:
# Agrupamos por casa y género y contamos los personajes de cada grupo.
personajes_por_casa_genero = (
    df_hogwarts
    .groupby(["House", "Gender"], dropna=False)
    .size()
)

# Mostramos el resultado.
personajes_por_casa_genero

House       Gender
Gryffindor  Female    12
            Male      26
Hufflepuff  Female     5
            Male       8
Ravenclaw   Female     9
            Male       9
Slytherin   Female     6
            Male      21
            NaN        1
dtype: int64

## Pregunta 3. ¿Qué tipos de hechizos aparecen con mayor frecuencia?

Analizamos la columna Type de la tabla de hechizos para conocer
qué categorías aparecen con mayor frecuencia.

Primero revisamos los valores distintos para comprobar cómo están
escritas las categorías antes de contarlas.

In [104]:
# Mostramos los tipos de hechizos registrados, sin repetirlos.
df_hechizos["Type"].unique()

array(['Charm', 'Charm, Conjuration', 'Spell',
       'Healing spell, Vanishment', 'Hex', 'Jinx', 'Curse',
       'Transportation', 'Conjuration', 'Transfiguration',
       'Conjuration, Charm', 'Healing spell, Charm', 'Transformation',
       'Dark charm', 'Charm,\xa0Counter-charm', 'Hex ',
       'Transfiguration,\xa0Jinx', 'Transfiguration\xa0Jinx',
       'Transfiguration\xa0spell', 'Healing Spell, Charm, Conjuration',
       'Counter-spell', 'Dark Arts', 'Charm ', 'Curse,\xa0Jinx',
       'Counter-jinx', 'Conjuration,\xa0Jinx', 'Counter-charm',
       'Patented\xa0Charm', 'Hex,\xa0Counter-curse', 'Untransfiguration',
       'Charm,\xa0Transfiguration', 'Transfiguration\xa0jinx',
       'Jinx,\xa0Hex', 'Vanishment', 'Healing spell, Counter-curse'],
      dtype=object)

Aquí hemos encontrado problemas de escritura y hechizos con varios tipos:

- "Hex" y "Hex "	El segundo tiene un espacio al final. Pandas los contaría por separado.  
- \xa0	Representa un espacio especial, llamado espacio de no separación.  
- "Charm, Conjuration"	Un hechizo tiene dos tipos registrados.  
"Conjuration, Charm"	Contiene los mismos tipos, escritos en otro orden.  
- "Transfiguration Jinx"	No tiene coma. Necesita revisión antes de decidir si son uno o varios tipos.  

Creamos una columna llamada Type_clean para sustituir esos espacios
especiales por espacios normales y eliminar los espacios del principio
y del final.

`.str.replace("\xa0", " ", regex=False)`	Sustituye cada espacio especial por uno normal.
`regex=False` indica que buscamos el texto literal.
`.str.strip()`	Elimina espacios al principio y al final de cada texto.

In [105]:
# Creamos una columna con los espacios normalizados.
df_hechizos["Type_clean"] = (
    df_hechizos["Type"]
    .str.replace("\xa0", " ", regex=False)
    .str.strip()
)

# Revisamos las etiquetas después de limpiar los espacios.
df_hechizos["Type_clean"].unique()

array(['Charm', 'Charm, Conjuration', 'Spell',
       'Healing spell, Vanishment', 'Hex', 'Jinx', 'Curse',
       'Transportation', 'Conjuration', 'Transfiguration',
       'Conjuration, Charm', 'Healing spell, Charm', 'Transformation',
       'Dark charm', 'Charm, Counter-charm', 'Transfiguration, Jinx',
       'Transfiguration Jinx', 'Transfiguration spell',
       'Healing Spell, Charm, Conjuration', 'Counter-spell', 'Dark Arts',
       'Curse, Jinx', 'Counter-jinx', 'Conjuration, Jinx',
       'Counter-charm', 'Patented Charm', 'Hex, Counter-curse',
       'Untransfiguration', 'Charm, Transfiguration',
       'Transfiguration jinx', 'Jinx, Hex', 'Vanishment',
       'Healing spell, Counter-curse'], dtype=object)

Algunos hechizos tienen varios tipos separados por comas.
Separamos esas etiquetas para contar cada tipo individualmente.

También unificamos las mayúsculas y minúsculas para que expresiones
como Healing spell y Healing Spell se cuenten juntas.

Conservamos las etiquetas sin comas como categorías independientes.
Esta decisión evita interpretar combinaciones que requieren revisión.

Un hechizo puede pertenecer a varios tipos, por lo que la suma de
los conteos puede superar el número total de hechizos.

In [106]:
# Separamos los tipos indicados mediante comas.
tipos_individuales = (
    df_hechizos["Type_clean"]
    .str.split(",") # Divide cada texto por las comas y devuelve una lista.
    .explode() # Convierte cada elemento de la lista en una fila separada.
    .str.strip() # Elimina los espacios que quedan alrededor de cada elemento.
    .str.lower() # Convierte el texto a minúsculas
)

# Contamos cuántos hechizos tienen cada tipo registrado.
hechizos_por_tipo = tipos_individuales.value_counts()

# Mostramos los resultados de mayor a menor frecuencia.
hechizos_por_tipo

,count
Type_clean,
charm,160
jinx,33
curse,27
transfiguration,23
hex,22
conjuration,13
spell,11
healing spell,6
counter-charm,3


## Pregunta 4 ¿Qué especies son más comunes entre los personajes con una casa de Hogwarts registrada?

Analizamos la columna Species para conocer qué especies
aparecen con mayor frecuencia entre los personajes seleccionados.

Primero revisamos las etiquetas para detectar posibles diferencias
de escritura antes de realizar el conteo.

In [110]:
# Conslultamos las especies registradas entre los personajes seleccionados.
df_personajes['Species'].unique()

array(['Human', 'Half-Human/Half-Giant', 'Werewolf',
       'Human\xa0(Werewolf\xa0traits)', 'Human(goblin ancestry)', 'Ghost',
       'Centaur', 'Human\xa0', 'Human\xa0(Metamorphmagus)', 'House elf'],
      dtype=object)

La columna Species contiene espacios especiales representados como \xa0.
Los sustituimos por espacios normales y eliminamos los espacios
del principio y del final.

Conservamos las categorías que incluyen características adicionales
o ascendencia, sin agruparlas automáticamente con Human.

Contamos las categorías resultantes entre los personajes que tienen
una casa de Hogwarts registrada.

In [115]:
# Preparamos las etiquetas sin modificar la columna original.
especies_limpias = (
    df_hogwarts["Species"]
    .str.replace("\xa0", " ", regex=False)
    .str.strip()
)

# Contamos los personajes de cada categoría.
personajes_por_especie = especies_limpias.value_counts()

# Mostramos el resultado.
personajes_por_especie

,count
Species,
Human,87
Ghost,5
Half-Human/Half-Giant,1
Werewolf,1
Human (Werewolf traits),1
Human(goblin ancestry),1
Human (Metamorphmagus),1


## Pregunta 5. ¿Qué nivel de dificultad es más frecuente entre las pociones
## que tienen ese dato registrado?



Exploramos la columna Difficulty level para comparar los niveles
de dificultad registrados.

Primero revisamos sus valores distintos para detectar posibles
diferencias de escritura o etiquetas que necesiten preparación.

Sabemos que solo 25 de las 72 pociones tienen este dato disponible.

In [116]:
# Mostramos los niveles de dificultad registrados, sin repetirlos.
df_pociones["Difficulty level"].unique()

array(['Advanced', nan, 'Beginner', 'Simple to moderate', 'O.W.L. level',
       'Above\xa0second year', 'Moderate to advanced', 'Moderate'],
      dtype=object)

La columna Difficulty level contiene distintas etiquetas de dificultad
y valores ausentes.

Limpiamos los espacios especiales y los espacios de los extremos.
Conservamos las etiquetas completas: Simple to moderate representa
un rango de dificultad y no debe dividirse en dos categorías.

Después contamos cuántas pociones tienen registrado cada nivel.
El conteo excluye los valores ausentes y describe las 25 pociones
con dificultad disponible.

In [117]:
dificultades_limpias = (
    df_pociones["Difficulty level"]
    .str.replace("\xa0", " ", regex=False)
    .str.strip()
)

# contamos las pociones de cada nivel.
pociones_por_dificultad = dificultades_limpias.value_counts()

# Mostramos el resultado.
pociones_por_dificultad

,count
Difficulty level,
Advanced,14
Beginner,6
Simple to moderate,1
O.W.L. level,1
Above second year,1
Moderate to advanced,1
Moderate,1
